# Setup

In [1]:
import copy
import warnings
import random
import numpy as np
import time
import pickle
import torch
import torch.nn as nn
from itertools import product
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, average_precision_score, brier_score_loss
from sklearn.svm import SVC, LinearSVC
from sklearn.exceptions import ConvergenceWarning
from sktime.transformations.panel.catch22 import Catch22
from sklearn.impute import SimpleImputer
from joblib import Parallel, delayed
from scipy.special import expit

In [2]:
# Random state seed
RAND_STATE = 42

In [3]:
# Load the normalized data
output_dir = Path('../processed/')
output_dir = Path("/kaggle/input/datasets/brocklarson/normalized/") # TODO: change back after done with kaggle

data = np.load(output_dir / 'normalized_split.npz')
X_train_norm = data['X_train']
X_val_norm = data['X_val']
X_test_norm = data['X_test']
y_train = data['y_train']
y_val = data['y_val']
y_test = data['y_test']

# with open(output_dir / 'norm_params.pkl', 'rb') as f_in:
#     norm_params = pickle.load(f_in)

print("Pre-processed data loaded successfully")

Pre-processed data loaded successfully


In [4]:
for name, X, y in [("TRAINING", X_train_norm, y_train),
                   ("VALIDATION", X_val_norm, y_val),
                   ("TEST", X_test_norm, y_test)]:
    n_sep = int((y == 1).sum())
    n_nsep = int((y == 0).sum())

    print("================")
    print(f"{name} DATA")
    print("================")
    print(f"X shape:    {X.shape}")
    print(f"y shape:    {y.shape}")
    print(f"NSEP count: {n_nsep}")
    print(f"SEP count:  {n_sep}")
    print(f"Ratio:      {n_nsep / n_sep:.2f}:1")
    print()

TRAINING DATA
X shape:    (12455, 288, 10)
y shape:    (12455,)
NSEP count: 12337
SEP count:  118
Ratio:      104.55:1

VALIDATION DATA
X shape:    (1780, 288, 10)
y shape:    (1780,)
NSEP count: 1763
SEP count:  17
Ratio:      103.71:1

TEST DATA
X shape:    (3559, 288, 10)
y shape:    (3559,)
NSEP count: 3525
SEP count:  34
Ratio:      103.68:1



In [5]:
# Flatten 
X_train_flat = X_train_norm.reshape(X_train_norm.shape[0], -1)
X_val_flat = X_val_norm.reshape(X_val_norm.shape[0], -1)
X_test_flat = X_test_norm.reshape(X_test_norm.shape[0], -1)

# Evaluation Metrics

In [6]:
def true_skill_statistic(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    # Guard against divide by zero
    if (tp + fn == 0 or
        tn + fp == 0):
        return 0.0

    tss = tp / (tp + fn) - fp / (tn + fp)
    
    return float(tss)

def heidke_skill_score(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    numerator = 2 * ((tp * tn) - (fp * fn))
    denominator = ((tp + fn) * (fn + tn)) + ((tn + fp) * (tp + fp))

    # Guard against divide by zero
    if denominator == 0:
        return 0.0
        
    return float(numerator / denominator)

def brier_skill_score(y_true, y_prob):
    base_rate = np.mean(y_true)
    brier_ref = brier_score_loss(y_true, np.full(len(y_true), base_rate))
    return 1 - brier_score_loss(y_true, y_prob) / brier_ref

def evaluate_model_metrics( 
        y_actual, 
        y_pred, 
        y_prob, 
        model_name="Model"
    ):    
    conf_matrix = confusion_matrix(y_actual, y_pred)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_actual, y_pred),
        "Precision": precision_score(y_actual, y_pred, zero_division=0),
        "Recall": recall_score(y_actual, y_pred, zero_division=0),
        "F1-Score": f1_score(y_actual, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_actual, y_prob),
        "PR-AUC": average_precision_score(y_actual, y_prob),
        "Brier": brier_score_loss(y_actual, y_prob),
        "BSS" : brier_skill_score(y_actual, y_prob),
        "TSS": true_skill_statistic(conf_matrix),
        "HSS2": heidke_skill_score(conf_matrix),
        "Confusion Matrix": conf_matrix
    }
    
    return metrics

def print_metrics(metrics):
    print("===================")
    print("PERFORMANCE METRICS")
    print("===================")
    for name, val in metrics.items():
        formatted_val = round(val,5) if isinstance(val, (float, np.floating)) else val   
        print(f"{name:<17}: {formatted_val}")
    print()

# Baseline Classifications

In [7]:
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Logistic Regression

In [8]:
def best_tss_threshold(y_true, y_prob):
    fpr, tpr, threshold = roc_curve(y_true, y_prob)
    tss = tpr - fpr
    i = np.argmax(tss)
    return threshold[i], tss[i]

N_VAL_SEEDS = 3  
N_FINAL_RUNS = 3 

# Grid search strategy on different C and class weights to see what provides best tss on validation set
results = []
for C in [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]:
    for class_weight in ['balanced', {0: 1, 1: 10}, {0: 1, 1: 30}]:

        # Run the same config over several seeds and average the results 
        tss_list = [] 
        for s in range(N_VAL_SEEDS): 
            model = LogisticRegression(
                C=C, 
                class_weight=class_weight, 
                max_iter=2000,
                random_state=RAND_STATE + s 
            )

            # Train model with new parameters
            model.fit(X_train_flat, y_train)
            prob = model.predict_proba(X_val_flat)[:, 1]

            # Find which threshold produces best TSS
            threshold, tss = best_tss_threshold(y_val, prob)
            tss_list.append(tss) 

        mean_tss = float(np.mean(tss_list))
        results.append((mean_tss, C, class_weight)) 
        print(f"C={C:<7} class_weight={str(class_weight):<15} mean TSS={mean_tss:.4f}  std={np.std(tss_list):.4f}") 

# Find what test produced the best mean TSS result on validation set
best_tss, best_C, best_class_weight = max(results, key=lambda r: r[0])
print(f"\nBest: C={best_C}, class_weight={best_class_weight}, mean val TSS={best_tss:.5f}\n")

# Re-run best config on the validation set over several seeds and report each run
def fit_logreg(seed): 
    m = LogisticRegression(  
        C=best_C,  
        class_weight=best_class_weight,  
        max_iter=2000,  
        random_state=seed 
    ) 
    m.fit(X_train_flat, y_train)  
    return m 

logreg_val_tss = []  
for s in range(N_VAL_SEEDS): 
    logreg_model = fit_logreg(RAND_STATE + s)  

    # Make predictions using a threshold tuned on the validation set
    y_val_prob = logreg_model.predict_proba(X_val_flat)[:, 1]
    thr, _ = best_tss_threshold(y_val, y_val_prob) 
    y_val_pred = (y_val_prob >= thr).astype(int)  

    logreg_results = evaluate_model_metrics(
        y_actual=y_val,
        y_pred=y_val_pred,
        y_prob=y_val_prob,
        model_name=f"Logistic Regression (Val) seed {RAND_STATE + s}" 
    )
    logreg_results["threshold"] = thr  
    print_metrics(logreg_results)
    logreg_val_tss.append(logreg_results["TSS"])  
    
print(f"Val TSS over {N_VAL_SEEDS} seeds: mean={np.mean(logreg_val_tss):.4f}  std={np.std(logreg_val_tss):.4f}") 

C=1e-06   class_weight=balanced        mean TSS=0.6320  std=0.0000
C=1e-06   class_weight={0: 1, 1: 10}   mean TSS=0.6117  std=0.0000
C=1e-06   class_weight={0: 1, 1: 30}   mean TSS=0.6303  std=0.0000
C=1e-05   class_weight=balanced        mean TSS=0.7695  std=0.0000
C=1e-05   class_weight={0: 1, 1: 10}   mean TSS=0.6620  std=0.0000
C=1e-05   class_weight={0: 1, 1: 30}   mean TSS=0.7203  std=0.0000
C=0.0001  class_weight=balanced        mean TSS=0.5788  std=0.0000
C=0.0001  class_weight={0: 1, 1: 10}   mean TSS=0.6795  std=0.0000
C=0.0001  class_weight={0: 1, 1: 30}   mean TSS=0.5817  std=0.0000
C=0.001   class_weight=balanced        mean TSS=0.4827  std=0.0000
C=0.001   class_weight={0: 1, 1: 10}   mean TSS=0.4737  std=0.0000
C=0.001   class_weight={0: 1, 1: 30}   mean TSS=0.4776  std=0.0000
C=0.01    class_weight=balanced        mean TSS=0.4566  std=0.0000
C=0.01    class_weight={0: 1, 1: 10}   mean TSS=0.4430  std=0.0000
C=0.01    class_weight={0: 1, 1: 30}   mean TSS=0.4305  std=0.

In [9]:
# Run on test set using tuned parameters, over several runs
logreg_test_tss = [] 
for run in range(N_FINAL_RUNS): 
    print(f"Run {run + 1}...", flush=True)  
    logreg_model = fit_logreg(RAND_STATE + 100 + run) 

    # Threshold tuned on the validation set for this run
    thr, _ = best_tss_threshold(y_val, logreg_model.predict_proba(X_val_flat)[:, 1]) 

    y_test_prob = logreg_model.predict_proba(X_test_flat)[:, 1]
    y_test_pred = (y_test_prob >= thr).astype(int)

    logreg_test_results = evaluate_model_metrics(
        y_actual=y_test,
        y_pred=y_test_pred,
        y_prob=y_test_prob,
        model_name=f"Logistic Regression (Test) run {run + 1}" 
    )
    logreg_test_results["threshold"] = thr  
    print_metrics(logreg_test_results)
    logreg_test_tss.append(logreg_test_results["TSS"]) 

print(f"Test TSS over {N_FINAL_RUNS} runs: mean={np.mean(logreg_test_tss):.4f}  std={np.std(logreg_test_tss):.4f}") 

Run 1...
PERFORMANCE METRICS
Model            : Logistic Regression (Test) run 1
Accuracy         : 0.86794
Precision        : 0.04008
Recall           : 0.55882
F1-Score         : 0.0748
ROC-AUC          : 0.73727
PR-AUC           : 0.06977
Brier            : 0.14836
BSS              : -14.67952
TSS              : 0.42975
HSS2             : 0.05801
Confusion Matrix : [[3070  455]
 [  15   19]]
threshold        : 0.48182

Run 2...
PERFORMANCE METRICS
Model            : Logistic Regression (Test) run 2
Accuracy         : 0.86794
Precision        : 0.04008
Recall           : 0.55882
F1-Score         : 0.0748
ROC-AUC          : 0.73727
PR-AUC           : 0.06977
Brier            : 0.14836
BSS              : -14.67952
TSS              : 0.42975
HSS2             : 0.05801
Confusion Matrix : [[3070  455]
 [  15   19]]
threshold        : 0.48182

Run 3...
PERFORMANCE METRICS
Model            : Logistic Regression (Test) run 3
Accuracy         : 0.86794
Precision        : 0.04008
Recall       

### XGBoost

In [10]:
rng = random.Random(RAND_STATE) # Define rng here so random state in build_model doesn't reset other random choices
K = 30 # Number of random sampling

learn_rate_opts = [0.1, 0.05, 0.02]
spw_opts = [1, 10, 30, 104]
depth_opts = [1, 2, 3, 4]
mcw_opts = [1, 3, 5, 10, 20]
cbt_opts = [0.1, 0.3, 0.5]

# Get random sample using grid to avoid duplicates
grid = list(product(learn_rate_opts, spw_opts, depth_opts, mcw_opts, cbt_opts))
sampled = rng.sample(grid, K)

N_VAL_SEEDS = 3  
N_FINAL_RUNS = 3 

results_xgb = []
for learn_rate, scale_pos_weight, max_depth, min_child_weight, colsample_bytree in sampled:

    # Run the same config over several seeds and average the results
    tss_list, iter_list = [], [] 
    for s in range(N_VAL_SEEDS):  
        model = XGBClassifier(
            scale_pos_weight=scale_pos_weight,  # Weights class imbalance
            max_depth=max_depth,                # Decision tree max depth
            min_child_weight=min_child_weight,  # Min sum of weight a leaf must have to exist
            n_estimators=1000,                  # No. of sequential decision trees (upper bound)
            learning_rate=learn_rate,
            subsample=0.7,                      # Random subsample tree is trained on
            colsample_bytree=colsample_bytree,  # Random percent of cols each tree sees
            reg_lambda=10,                      # L2 penalty on leaf values
            eval_metric="aucpr",
            early_stopping_rounds=50,           # Early stopping condition
            random_state=RAND_STATE + s, 
            n_jobs=-1,
        )

        # Train model with new parameters
        model.fit(
            X_train_flat, y_train,
            eval_set=[(X_val_flat, y_val)],
            verbose=False
        )

        # Find which threshold produces best TSS
        prob = model.predict_proba(X_val_flat)[:, 1]
        thr, tss = best_tss_threshold(y_val, prob)
        tss_list.append(tss) 
        iter_list.append(model.best_iteration) 

    mean_tss = float(np.mean(tss_list))
    mean_iter = int(round(np.mean(iter_list)))
    results_xgb.append((mean_tss, learn_rate, scale_pos_weight, max_depth, min_child_weight, colsample_bytree, mean_iter))
    print(f"lr={learn_rate:<5} spw={scale_pos_weight:<4} md={max_depth} mcw={min_child_weight} cbt={colsample_bytree} trees={mean_iter:<4} mean val TSS={mean_tss:.5f}  std={np.std(tss_list):.5f}")

# Find what test produced the best mean TSS result on validation set
(best_tss, best_lr, best_spw, best_md, best_mcw, best_cbt, best_iter) = max(results_xgb, key=lambda r: r[0])
print(f"\nBest: lr={best_lr}, spw={best_spw}, md={best_md}, mcw={best_mcw}, cbt={best_cbt}, trees={best_iter + 1}, mean val TSS={best_tss:.5f}\n")

# Final model builder for the best config (fixed number of trees)
def fit_best_xgb(seed):
    m = XGBClassifier(  
        scale_pos_weight=best_spw,
        max_depth=best_md,
        min_child_weight=best_mcw,
        n_estimators=best_iter + 1,        
        learning_rate=best_lr,
        subsample=0.7,
        colsample_bytree=best_cbt,
        reg_lambda=10,
        random_state=seed, 
        n_jobs=-1,
    )
    m.fit(X_train_flat, y_train) 
    return m 

# Re-run best config on the validation set over several seeds and report each run  
xgb_val_tss = []
for s in range(N_VAL_SEEDS):  
    xgb_model = fit_best_xgb(RAND_STATE + s) 

    # Make predictions using a threshold tuned on the validation set
    y_val_prob_xgb = xgb_model.predict_proba(X_val_flat)[:, 1]
    thr, _ = best_tss_threshold(y_val, y_val_prob_xgb) 
    y_val_pred_xgb = (y_val_prob_xgb >= thr).astype(int)

    xgb_results = evaluate_model_metrics(
        y_actual=y_val,
        y_pred=y_val_pred_xgb,
        y_prob=y_val_prob_xgb,
        model_name=f"XGBoost (Val) seed {RAND_STATE + s}"
    )
    xgb_results["threshold"] = thr 
    print_metrics(xgb_results)
    xgb_val_tss.append(xgb_results["TSS"])  

print(f"Val TSS over {N_VAL_SEEDS} seeds: mean={np.mean(xgb_val_tss):.4f}  std={np.std(xgb_val_tss):.4f}") 

lr=0.02  spw=30   md=4 mcw=10 cbt=0.1 trees=5    mean val TSS=0.54196  std=0.06302
lr=0.1   spw=10   md=4 mcw=10 cbt=0.1 trees=10   mean val TSS=0.62757  std=0.03531
lr=0.1   spw=1    md=2 mcw=10 cbt=0.3 trees=93   mean val TSS=0.69409  std=0.00965
lr=0.05  spw=1    md=3 mcw=10 cbt=0.5 trees=34   mean val TSS=0.75170  std=0.01019
lr=0.05  spw=1    md=1 mcw=10 cbt=0.3 trees=55   mean val TSS=0.69844  std=0.01973
lr=0.1   spw=104  md=4 mcw=3 cbt=0.1 trees=13   mean val TSS=0.60507  std=0.01667
lr=0.1   spw=30   md=2 mcw=5 cbt=0.3 trees=46   mean val TSS=0.67119  std=0.02510
lr=0.1   spw=10   md=3 mcw=20 cbt=0.5 trees=66   mean val TSS=0.64402  std=0.01053
lr=0.02  spw=104  md=3 mcw=1 cbt=0.5 trees=49   mean val TSS=0.68702  std=0.01020
lr=0.02  spw=10   md=2 mcw=3 cbt=0.1 trees=8    mean val TSS=0.62820  std=0.01921
lr=0.1   spw=10   md=2 mcw=20 cbt=0.5 trees=4    mean val TSS=0.64913  std=0.00512
lr=0.02  spw=30   md=1 mcw=3 cbt=0.3 trees=149  mean val TSS=0.69439  std=0.00233
lr=0.05  

In [ ]:
# Run on test set using tuned parameters, over several runs 
xgb_test_tss = []  
for run in range(N_FINAL_RUNS):
    print(f"Run {run + 1}", flush=True) 
    xgb_model = fit_best_xgb(RAND_STATE + 100 + run)

    # Threshold tuned on the validation set for this run
    thr, _ = best_tss_threshold(y_val, xgb_model.predict_proba(X_val_flat)[:, 1])

    y_test_prob_xgb = xgb_model.predict_proba(X_test_flat)[:, 1]
    y_test_pred_xgb = (y_test_prob_xgb >= thr).astype(int)

    xgb_test_results = evaluate_model_metrics(
        y_actual=y_test,
        y_pred=y_test_pred_xgb,
        y_prob=y_test_prob_xgb,
        model_name=f"XGBoost (Test) run {run + 1}"
    )
    xgb_test_results["threshold"] = thr
    print_metrics(xgb_test_results)
    xgb_test_tss.append(xgb_test_results["TSS"])

print(f"Test TSS over {N_FINAL_RUNS} runs: mean={np.mean(xgb_test_tss):.4f}  std={np.std(xgb_test_tss):.4f}")

Run 1...
PERFORMANCE METRICS
Model            : XGBoost (Test) run 1
Accuracy         : 0.87749
Precision        : 0.05727
Recall           : 0.76471
F1-Score         : 0.10656
ROC-AUC          : 0.87717
PR-AUC           : 0.15024
Brier            : 0.009
BSS              : 0.04916
TSS              : 0.64329
HSS2             : 0.09039
Confusion Matrix : [[3097  428]
 [   8   26]]
threshold        : 0.007360000163316727

Run 2...
PERFORMANCE METRICS
Model            : XGBoost (Test) run 2
Accuracy         : 0.89407
Precision        : 0.06582
Recall           : 0.76471
F1-Score         : 0.12121
ROC-AUC          : 0.89429
PR-AUC           : 0.17964
Brier            : 0.00899
BSS              : 0.05036
TSS              : 0.66003
HSS2             : 0.10548
Confusion Matrix : [[3156  369]
 [   8   26]]
threshold        : 0.009180000051856041

Run 3...
PERFORMANCE METRICS
Model            : XGBoost (Test) run 3
Accuracy         : 0.89435
Precision        : 0.06599
Recall           : 0.76471


### GRU

In [12]:
warnings.filterwarnings("ignore")

# GRU model───────────────────────────────────────────────
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()

        self.gru = nn.GRU(
            input_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.bn = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]         
        out = self.bn(out)
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.dropout2(out)
        return self.fc2(out).squeeze(-1)


# GRU hyperparameter grid─────────────────────────────────
GRU_GRID = [
    # Small models
    {"units": 32,  "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 64},
    {"units": 32,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},

    # Medium models
    {"units": 64,  "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 64},
    {"units": 64,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},

    # Larger single-layer models
    {"units": 128, "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 32},
    {"units": 128, "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 32},

    # Two-layer models
    {"units": 64,  "dropout": 0.1, "layers": 2, "lr": 0.001,  "batch_size": 64},
    {"units": 128, "dropout": 0.3, "layers": 2, "lr": 0.001,  "batch_size": 32},

    # Lower learning rate variants
    {"units": 64,  "dropout": 0.3, "layers": 2, "lr": 0.0005, "batch_size": 64},
    {"units": 128, "dropout": 0.3, "layers": 2, "lr": 0.0005, "batch_size": 32},
]

def gru_label(p):
    return (
        f"units={p['units']}  "
        f"layers={p['layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )

# Train GRU and evaluate on a provided test split──────────
def train_and_evaluate_gru(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=50,
    patience=5,
    seed=RAND_STATE
):
    torch.manual_seed(seed)
    
    # Convert numpy arrays to tensors
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    # Build model
    input_size = X_train.shape[2]

    model = GRUModel(
        input_size=input_size,
        hidden_size=params["units"],
        num_layers=params["layers"],
        dropout=params["dropout"]
    ).to(device)

    # Weighted BCE loss for class imbalance
    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for _ in range(max_epochs):
        model.train()

        # Shuffle training samples each epoch
        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            if len(idx) < 2:                        
                continue    

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        # Validation loss for early stopping
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_count = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    # Load best validation-loss model
    if best_state is not None:
        model.load_state_dict(best_state)

    # Tune threshold on val                                                
    model.eval()
    with torch.no_grad():                                                  
        y_val_prob = torch.sigmoid(model(X_va)).cpu().numpy()              
    thr, _ = best_tss_threshold(y_val.astype(int), y_val_prob)             

    # Prediction on the eval split
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= thr).astype(int) 
    infer_time = time.time() - t0

    metrics = evaluate_model_metrics(
        y_actual=y_test.astype(int),                
        y_pred=y_pred,
        y_prob=y_prob,   
        model_name="GRU"
    )
    metrics["threshold"] = thr
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

In [ ]:
# ══════════════════════════════════════════════════════════════
# RUN GRU EXPERIMENTS
# Tune on validation set, then evaluate best params on test set
# ══════════════════════════════════════════════════════════════
N_FINAL_RUNS = 3 
N_VAL_SEEDS = 3 

print(f"{'═' * 60}")
print("  Classifier : GRU")
print(f"{'═' * 60}")

# Sanity check: GRU expects (samples, 288, 10)
print("X_train:", X_train_norm.shape, " X_val:", X_val_norm.shape, " X_test:", X_test_norm.shape)

class_ratio = round((y_train == 0).sum() / (y_train == 1).sum(), 1)
print(f"Train class ratio 0:1 = {class_ratio}:1\n")

# Hyperparameter search on validation set only───────────────────
print(f"Searching {len(GRU_GRID)} GRU combinations on validation set...\n")

best_params = None
best_tss = -999
best_val_metrics = None

for i, params in enumerate(GRU_GRID):
    print(f"[{i + 1}/{len(GRU_GRID)}] {gru_label(params)}", flush=True)

    # Run the same config over several seeds and average the results 
    val_runs = [] 
    for s in range(N_VAL_SEEDS):  
        # Validation set is used as the evaluation split during tuning
        metrics, _ = train_and_evaluate_gru(
            params,
            X_train_norm, y_train,
            X_val_norm,   y_val,
            X_val_norm,   y_val,
            class_ratio=class_ratio,
            seed=RAND_STATE + s  
        )
        val_runs.append(metrics)  

        print(  
            f"    seed={RAND_STATE + s}  Val TSS={metrics['TSS']:.4f}  " 
            f"F1={metrics['F1-Score']:.4f}  "
            f"Recall={metrics['Recall']:.4f}  " 
            f"Thr={metrics['threshold']:.4f}  "  
            f"Train={metrics['train_time']:.1f}s"  
        ) 

    val_tss = [m["TSS"] for m in val_runs] 
    mean_val_tss = float(np.mean(val_tss))  
    std_val_tss = float(np.std(val_tss)) 
    print(f"    Mean Val TSS={mean_val_tss:.4f}  std={std_val_tss:.4f}\n") 

    if mean_val_tss > best_tss: 
        best_tss = mean_val_tss
        best_params = copy.deepcopy(params)
        best_val_metrics = val_runs  
        best_val_std = std_val_tss 

print("Best GRU hyperparameters selected from validation set:")
print(f"  Params  : {gru_label(best_params)}")
print(f"  Mean val TSS over {N_VAL_SEEDS} seeds: {best_tss:.4f}  std={best_val_std:.4f}")
for r, m in enumerate(best_val_metrics): 
    print(f"  Val seed {RAND_STATE + r}:")  
    print_metrics(m) 

# Final runs on the test set with the best hyperparameters───────
print(f"\nRunning {N_FINAL_RUNS} final runs with the best hyperparameters\n")
print("─" * 70)
print(f"Params  : {gru_label(best_params)}")
print(f"Ratio   : {class_ratio}:1")
print("─" * 70)

metrics_list = []

for run in range(N_FINAL_RUNS):
    print(f"Run {run + 1}", flush=True)

    metrics, _ = train_and_evaluate_gru(
        best_params,
        X_train_norm, y_train,
        X_val_norm,   y_val,
        X_test_norm,  y_test,
        class_ratio=class_ratio,
        seed=RAND_STATE + 100 + run  
    )

    metrics_list.append(metrics)

    print_metrics(metrics)

tss_runs = [m["TSS"] for m in metrics_list]        
print(f"Test TSS over {N_FINAL_RUNS} runs: mean={np.mean(tss_runs):.4f}  std={np.std(tss_runs):.4f}")  

print("\nGRU experiment complete.")

════════════════════════════════════════════════════════════
  Classifier : GRU
════════════════════════════════════════════════════════════
X_train: (12455, 288, 10)  X_val: (1780, 288, 10)  X_test: (3559, 288, 10)
Train class ratio 0:1 = 104.6:1

Searching 10 GRU combinations on validation set...

[1/10] units=32  layers=1  drop=0.1  lr=0.001  bs=64
    seed=42  Val TSS=0.6555  F1=0.0694  Recall=0.8824  Thr=0.4973  Train=6.8s
    seed=43  Val TSS=0.7260  F1=0.1379  Recall=0.8235  Thr=0.6711  Train=3.6s
    seed=44  Val TSS=0.6953  F1=0.1089  Recall=0.8235  Thr=0.5768  Train=4.2s
    Mean Val TSS=0.6923  std=0.0289

[2/10] units=32  layers=1  drop=0.3  lr=0.001  bs=64
    seed=42  Val TSS=0.6731  F1=0.0748  Recall=0.8824  Thr=0.5075  Train=3.2s
    seed=43  Val TSS=0.6863  F1=0.1026  Recall=0.8235  Thr=0.5778  Train=4.2s
    seed=44  Val TSS=0.6602  F1=0.0878  Recall=0.8235  Thr=0.5036  Train=4.1s
    Mean Val TSS=0.6732  std=0.0107

[3/10] units=64  layers=1  drop=0.1  lr=0.001  bs=6

### SVM 

In [ ]:
# ══════════════════════════════════════════════════════════════
# CATCH22 FEATURE EXTRACTION (hybrid-normalized data only)
# ══════════════════════════════════════════════════════════════

def _catch22_chunk(X_chunk: np.ndarray) -> np.ndarray:
    return Catch22(catch24=False).fit_transform(X_chunk).to_numpy()

def extract_catch22(X: np.ndarray, label: str = "", n_jobs: int = -1, chunk_size: int = 200) -> np.ndarray:
    n_samples, n_timepoints, n_channels = X.shape

    # Safety net: replace any non-finite values with the channel's median
    if not np.isfinite(X).all():
        print(f"    non-finite values in {label} — replacing with per-channel median")
        X = X.copy()
        for c in range(n_channels):
            col = X[:, :, c]                       # view, so edits land in X
            bad = ~np.isfinite(col)
            col[bad] = np.median(col[~bad])

    # sktime expects (samples, channels, timesteps)
    X_sktime = X.transpose(0, 2, 1).astype(np.float64)

    chunks = [X_sktime[i:i + chunk_size] for i in range(0, n_samples, chunk_size)]
    print(f"    {label} — {n_samples} samples in {len(chunks)} chunks ", flush=True)

    results = Parallel(n_jobs=n_jobs)(delayed(_catch22_chunk)(ch) for ch in chunks)
    X_feat = np.vstack(results)

    assert X_feat.shape == (n_samples, 22 * n_channels), f"Unexpected shape: {X_feat.shape}"
    return X_feat


# Extract features for each split──────────────────────────
print("Extracting Catch22 features ")
X_train_c22 = extract_catch22(X_train_norm, label="train")
X_val_c22   = extract_catch22(X_val_norm,   label="val")
X_test_c22  = extract_catch22(X_test_norm,  label="test")
print(f"train {X_train_c22.shape}  val {X_val_c22.shape}  test {X_test_c22.shape}")


# ══════════════════════════════════════════════════════════════
# IMPUTE — fit on train only, apply to val/test
# ══════════════════════════════════════════════════════════════

n_bad = [int((~np.isfinite(a)).sum()) for a in (X_train_c22, X_val_c22, X_test_c22)]

if any(n_bad):
    print(f"non-finite catch22 values — train: {n_bad[0]}  val: {n_bad[1]}  test: {n_bad[2]} → imputing")

    # Treat inf as missing too, so the imputer handles both
    X_train_c22, X_val_c22, X_test_c22 = [
        np.where(np.isfinite(a), a, np.nan) for a in (X_train_c22, X_val_c22, X_test_c22)
    ]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        imp = SimpleImputer(strategy="median", keep_empty_features=True)
        X_train_c22 = imp.fit_transform(X_train_c22)
        X_val_c22   = imp.transform(X_val_c22)
        X_test_c22  = imp.transform(X_test_c22)
else:
    print("✓ No NaN or Inf in catch22 features — no imputation needed")

X_train_c22, X_val_c22, X_test_c22 = [
    a.astype(np.float32) for a in (X_train_c22, X_val_c22, X_test_c22)
]

print(f"Ready: train {X_train_c22.shape}  val {X_val_c22.shape}  test {X_test_c22.shape}")


CLIP_Z = 10
X_train_c22, X_val_c22, X_test_c22 = [
    np.clip(a, -CLIP_Z, CLIP_Z).astype(np.float32)
    for a in (X_train_c22, X_val_c22, X_test_c22)
]

print(f"train col mean range: {X_train_c22.mean(0).min():.2g} to {X_train_c22.mean(0).max():.2g}")
print(f"train col std  range: {X_train_c22.std(0).min():.2g} to {X_train_c22.std(0).max():.2g}")
print(f"val   col max |z|   : {np.abs(X_val_c22).max():.1f}")

Extracting Catch22 features …
    train — 12455 samples in 63 chunks …
    val — 1780 samples in 9 chunks …
    test — 3559 samples in 18 chunks …
train (12455, 220)  val (1780, 220)  test (3559, 220)
non-finite catch22 values — train: 84320  val: 20276  test: 33898 → imputing
Ready: train (12455, 220)  val (1780, 220)  test (3559, 220)
train col mean range: -0.35 to 10
train col std  range: 0 to 3.8
val   col max |z|   : 10.0


In [15]:
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ══════════════════════════════════════════════════════════════
# SVM HYPERPARAMETER GRID
# Tune only on hybrid validation set
# ══════════════════════════════════════════════════════════════

SVM_GRID = [
    {"model": LinearSVC, "params": {"C": 0.001, "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.01, "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.05, "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.1,  "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.5,  "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 1.0,  "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 3.0,  "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.01,  "class_weight": {0: 1, 1: 10}, "max_iter": 10000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 0.01,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 0.1,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 1.0,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 10.0, "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 0.1,  "gamma": 0.001,   "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 0.1,  "gamma": "scale", "class_weight": {0: 1, 1: 10}, "cache_size": 2000}}
]


def combo_label(combo):
    ModelClass = combo["model"]
    params = combo["params"]

    cw = "balanced" if params.get("class_weight") == "balanced" else "none"

    if ModelClass == LinearSVC:
        return f"LinearSVC | C={params['C']} | cw={cw}"
    else:
        return f"RBF-SVC   | C={params['C']} | gamma={params['gamma']} | cw={cw}"


# ══════════════════════════════════════════════════════════════
# STEP 1: SEARCH BEST HYPERPARAMETERS ON HYBRID VALIDATION SET ONLY
# ══════════════════════════════════════════════════════════════

print("═" * 70)
print("SVM hyperparameter search on HYBRID validation set only")
print("═" * 70)

d_hybrid = {
    "X_train": X_train_c22, "y_train": y_train,
    "X_val":   X_val_c22,   "y_val":   y_val,
    "X_test":  X_test_c22,  "y_test":  y_test,
}

N_VAL_SEEDS = 3 
N_FINAL_RUNS = 3

# Fit one SVM with a given seed and evaluate on the given split 
def run_svm(ModelClass, params, seed, X_eval, y_eval, name):  
    model = ModelClass(**params, random_state=seed) 
    model.fit(d_hybrid["X_train"], d_hybrid["y_train"]) 
    y_pred = model.predict(X_eval) 
    y_prob = expit(model.decision_function(X_eval))
    return evaluate_model_metrics( 
        y_actual=y_eval.astype(int), 
        y_pred=y_pred,  
        y_prob=y_prob, 
        model_name=name
    ) 

best_tss = -np.inf
best_model_class = None
best_params = None
best_val_runs = None

for combo in SVM_GRID:
    ModelClass = combo["model"]
    params = copy.deepcopy(combo["params"])

    # Run the same config over several seeds and average the results
    t0 = time.time()
    val_runs = [
        run_svm(ModelClass, params, RAND_STATE + s, d_hybrid["X_val"], d_hybrid["y_val"], combo_label(combo)) 
        for s in range(N_VAL_SEEDS)
    ]
    elapsed = time.time() - t0

    val_tss = [m["TSS"] for m in val_runs] 
    mean_tss = float(np.mean(val_tss)) 

    print(
        f"{combo_label(combo):<55} "
        f"Mean Val TSS={mean_tss:.4f} (std={np.std(val_tss):.4f}) | "
        f"F1={np.mean([m['F1-Score'] for m in val_runs]):.4f} | "
        f"{elapsed:.1f}s"
    )

    if mean_tss > best_tss:  
        best_tss = mean_tss 
        best_model_class = ModelClass
        best_params = params
        best_val_runs = val_runs 

print("\nBest hyperparameters selected from hybrid validation set:")
print(f"  Model   : {best_model_class.__name__}")
print(f"  Params  : {best_params}")
print(f"  Mean val TSS over {N_VAL_SEEDS} seeds: {best_tss:.4f}  std={np.std([m['TSS'] for m in best_val_runs]):.4f}")  
for s, m in enumerate(best_val_runs): 
    print(f"  Val seed {RAND_STATE + s}:")
    print_metrics(m) 


# ══════════════════════════════════════════════════════════════
# STEP 2: TEST EVALUATION WITH THE BEST CONFIG OVER SEVERAL RUNS
# ══════════════════════════════════════════════════════════════

print(f"Running {N_FINAL_RUNS} final runs with the best hyperparameters...\n")
svm_test_tss = [] 
for run in range(N_FINAL_RUNS):
    print(f"Run {run + 1}...", flush=True) 
    svm_test_results = run_svm( 
        best_model_class, best_params, RAND_STATE + 100 + run, 
        d_hybrid["X_test"], d_hybrid["y_test"],
        f"{best_model_class.__name__} (test) run {run + 1}"
    )
    print_metrics(svm_test_results)
    svm_test_tss.append(svm_test_results["TSS"])

print(f"Test TSS over {N_FINAL_RUNS} runs: mean={np.mean(svm_test_tss):.4f}  std={np.std(svm_test_tss):.4f}")

══════════════════════════════════════════════════════════════════════
SVM hyperparameter search on HYBRID validation set only
══════════════════════════════════════════════════════════════════════
LinearSVC | C=0.001 | cw=balanced                       Mean Val TSS=0.6521 (std=0.0000) | F1=0.0685 | 3.8s
LinearSVC | C=0.01 | cw=balanced                        Mean Val TSS=0.6596 (std=0.0000) | F1=0.0875 | 8.7s
LinearSVC | C=0.05 | cw=balanced                        Mean Val TSS=0.5618 (std=0.0000) | F1=0.0848 | 16.7s
LinearSVC | C=0.1 | cw=balanced                         Mean Val TSS=0.4498 (std=0.0000) | F1=0.0738 | 27.0s
LinearSVC | C=0.5 | cw=balanced                         Mean Val TSS=0.2094 (std=0.0000) | F1=0.0435 | 66.4s
LinearSVC | C=1.0 | cw=balanced                         Mean Val TSS=0.2106 (std=0.0000) | F1=0.0438 | 181.7s
LinearSVC | C=3.0 | cw=balanced                         Mean Val TSS=0.2055 (std=0.0000) | F1=0.0424 | 197.5s
LinearSVC | C=0.01 | cw=none           

### MLP

In [16]:
# MLP model───────────────────────────────────────────────
class MLPModel(nn.Module):
    def __init__(self, input_size, hidden_sizes=(128,), dropout=0.3):
        super(MLPModel, self).__init__()

        layers = []
        in_dim = input_size
        for h in hidden_sizes:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        out = x.flatten(1)          # (batch, 288, C) -> (batch, 288 * C)
        return self.net(out).squeeze(-1)


# MLP hyperparameter grid─────────────────────────────────
MLP_GRID = [
    # Single hidden layer
    {"hidden": (64,),     "dropout": 0.3, "lr": 0.001,  "weight_decay": 1e-4, "batch_size": 64},
    {"hidden": (128,),    "dropout": 0.3, "lr": 0.001,  "weight_decay": 1e-4, "batch_size": 64},
    {"hidden": (128,),    "dropout": 0.1, "lr": 0.001,  "weight_decay": 1e-4, "batch_size": 64},
    {"hidden": (128,),    "dropout": 0.5, "lr": 0.001,  "weight_decay": 1e-3, "batch_size": 64},
    {"hidden": (256,),    "dropout": 0.5, "lr": 0.001,  "weight_decay": 1e-3, "batch_size": 64},

    # Two hidden layers
    {"hidden": (128, 32), "dropout": 0.3, "lr": 0.001,  "weight_decay": 1e-4, "batch_size": 64},
    {"hidden": (256, 64), "dropout": 0.3, "lr": 0.001,  "weight_decay": 1e-4, "batch_size": 32},
    {"hidden": (256, 64), "dropout": 0.1, "lr": 0.001,  "weight_decay": 1e-4, "batch_size": 32},
    {"hidden": (256, 64), "dropout": 0.5, "lr": 0.001,  "weight_decay": 1e-3, "batch_size": 32},

    # Lower learning rate variants
    {"hidden": (128,),    "dropout": 0.3, "lr": 0.0005, "weight_decay": 1e-4, "batch_size": 64},
    {"hidden": (256, 64), "dropout": 0.3, "lr": 0.0005, "weight_decay": 1e-3, "batch_size": 32},
]

def mlp_label(p):
    return (
        f"hidden={p['hidden']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"wd={p['weight_decay']}  "
        f"bs={p['batch_size']}"
    )

# Train MLP and evaluate on a provided test split──────────
def train_and_evaluate_mlp(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=50,
    patience=5,
    seed=RAND_STATE
):
    torch.manual_seed(seed)

    # Convert numpy arrays to tensors
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    # Build model (input is flattened timesteps * channels)
    input_size = X_train.shape[1] * X_train.shape[2]

    model = MLPModel(
        input_size=input_size,
        hidden_sizes=params["hidden"],
        dropout=params["dropout"]
    ).to(device)

    # Weighted BCE loss for class imbalance
    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"]
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for _ in range(max_epochs):
        model.train()

        # Shuffle training samples each epoch
        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            if len(idx) < 2:    # BatchNorm needs more than 1 sample
                continue

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        # Validation loss for early stopping
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_count = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    # Load best validation-loss model
    if best_state is not None:
        model.load_state_dict(best_state)

    # Tune threshold on val
    model.eval()
    with torch.no_grad():
        y_val_prob = torch.sigmoid(model(X_va)).cpu().numpy()
    thr, _ = best_tss_threshold(y_val.astype(int), y_val_prob)

    # Prediction on the eval split
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= thr).astype(int)
    infer_time = time.time() - t0

    metrics = evaluate_model_metrics(
        y_actual=y_test.astype(int),
        y_pred=y_pred,
        y_prob=y_prob,
        model_name="MLP"
    )
    metrics["threshold"] = thr
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

In [17]:
# ══════════════════════════════════════════════════════════════
# RUN MLP EXPERIMENTS
# Tune on validation set (multiple seeds), then evaluate best params on test set
# ══════════════════════════════════════════════════════════════
N_FINAL_RUNS = 3
N_VAL_SEEDS = 3

print(f"{'═' * 60}")
print("  Classifier : MLP")
print(f"{'═' * 60}")

print("X_train:", X_train_norm.shape, " X_val:", X_val_norm.shape, " X_test:", X_test_norm.shape)

class_ratio = round((y_train == 0).sum() / (y_train == 1).sum(), 1)
print(f"Train class ratio 0:1 = {class_ratio}:1\n")

# Hyperparameter search on validation set only───────────────────
print(f"Searching {len(MLP_GRID)} MLP combinations on validation set\n")

mlp_best_params = None
mlp_best_tss = -999
mlp_best_val_metrics = None

for i, params in enumerate(MLP_GRID):
    print(f"[{i + 1}/{len(MLP_GRID)}] {mlp_label(params)}", flush=True)

    # Run the same config over several seeds and average the results 
    val_runs = []  
    for s in range(N_VAL_SEEDS): 
        # Validation set is used as the evaluation split during tuning
        metrics, _ = train_and_evaluate_mlp(
            params,
            X_train_norm, y_train,
            X_val_norm,   y_val,
            X_val_norm,   y_val,
            class_ratio=class_ratio,
            seed=RAND_STATE + s  
        )
        val_runs.append(metrics)  

        print(  
            f"    seed={RAND_STATE + s}  Val TSS={metrics['TSS']:.4f}  "  
            f"F1={metrics['F1-Score']:.4f}  "  
            f"Recall={metrics['Recall']:.4f}  "
            f"Thr={metrics['threshold']:.4f}  "  
            f"Train={metrics['train_time']:.1f}s" 
        )  

    val_tss = [m["TSS"] for m in val_runs] 
    mean_val_tss = float(np.mean(val_tss))  
    std_val_tss = float(np.std(val_tss))  
    print(f"    Mean Val TSS={mean_val_tss:.4f}  std={std_val_tss:.4f}\n") 

    if mean_val_tss > mlp_best_tss: 
        mlp_best_tss = mean_val_tss 
        mlp_best_params = copy.deepcopy(params)
        mlp_best_val_metrics = val_runs
        mlp_best_val_std = std_val_tss 

print("Best MLP hyperparameters selected from validation set:")
print(f"  Params  : {mlp_label(mlp_best_params)}")
print(f"  Mean val TSS over {N_VAL_SEEDS} seeds: {mlp_best_tss:.4f}  std={mlp_best_val_std:.4f}")
for r, m in enumerate(mlp_best_val_metrics):
    print(f"  Val seed {RAND_STATE + r}:")
    print_metrics(m)

# Final runs on the test set with the best hyperparameters───────
print(f"\nRunning {N_FINAL_RUNS} final runs with the best hyperparameters...\n")
print("─" * 70)
print(f"Params  : {mlp_label(mlp_best_params)}")
print(f"Ratio   : {class_ratio}:1")
print("─" * 70)

mlp_metrics_list = []

for run in range(N_FINAL_RUNS):
    print(f"Run {run + 1}...", flush=True)

    metrics, _ = train_and_evaluate_mlp(
        mlp_best_params,
        X_train_norm, y_train,
        X_val_norm,   y_val,
        X_test_norm,  y_test,
        class_ratio=class_ratio,
        seed=RAND_STATE + 100 + run 
    )

    mlp_metrics_list.append(metrics)

    print_metrics(metrics)

mlp_tss_runs = [m["TSS"] for m in mlp_metrics_list]
print(f"Test TSS over {N_FINAL_RUNS} runs: mean={np.mean(mlp_tss_runs):.4f}  std={np.std(mlp_tss_runs):.4f}")

print("\nMLP experiment complete.")

════════════════════════════════════════════════════════════
  Classifier : MLP
════════════════════════════════════════════════════════════
X_train: (12455, 288, 10)  X_val: (1780, 288, 10)  X_test: (3559, 288, 10)
Train class ratio 0:1 = 104.6:1

Searching 11 MLP combinations on validation set

[1/11] hidden=(64,)  drop=0.3  lr=0.001  wd=0.0001  bs=64
    seed=42  Val TSS=0.7095  F1=0.1207  Recall=0.8235  Thr=0.5806  Train=4.6s
    seed=43  Val TSS=0.7239  F1=0.0769  Recall=0.9412  Thr=0.4097  Train=2.9s
    seed=44  Val TSS=0.7358  F1=0.0810  Recall=0.9412  Thr=0.4538  Train=2.3s
    Mean Val TSS=0.7231  std=0.0108

[2/11] hidden=(128,)  drop=0.3  lr=0.001  wd=0.0001  bs=64
    seed=42  Val TSS=0.6834  F1=0.1007  Recall=0.8235  Thr=0.4854  Train=4.3s
    seed=43  Val TSS=0.7523  F1=0.0874  Recall=0.9412  Thr=0.4946  Train=2.3s
    seed=44  Val TSS=0.7247  F1=0.0968  Recall=0.8824  Thr=0.4871  Train=2.3s
    Mean Val TSS=0.7201  std=0.0283

[3/11] hidden=(128,)  drop=0.1  lr=0.001  w

### LSTM

In [18]:
# LSTM model──────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.bn = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.bn(out)
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.dropout2(out)
        return self.fc2(out).squeeze(-1)


# LSTM hyperparameter grid────────────────────────────────
LSTM_GRID = [
    # Small models
    {"units": 32,  "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 64},
    {"units": 32,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},

    # Medium models
    {"units": 64,  "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 64},
    {"units": 64,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},

    # Larger single-layer models
    {"units": 128, "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 32},
    {"units": 128, "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 32},

    # Two-layer models
    {"units": 64,  "dropout": 0.1, "layers": 2, "lr": 0.001,  "batch_size": 64},
    {"units": 128, "dropout": 0.3, "layers": 2, "lr": 0.001,  "batch_size": 32},

    # Lower learning rate variants
    {"units": 64,  "dropout": 0.3, "layers": 2, "lr": 0.0005, "batch_size": 64},
    {"units": 128, "dropout": 0.3, "layers": 2, "lr": 0.0005, "batch_size": 32},
]

def lstm_label(p):
    return (
        f"units={p['units']}  "
        f"layers={p['layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )

# Train LSTM and evaluate on a provided test split─────────
def train_and_evaluate_lstm(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=50,
    patience=5,
    seed=RAND_STATE
):
    torch.manual_seed(seed)

    # Convert numpy arrays to tensors
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    # Build model
    input_size = X_train.shape[2]

    model = LSTMModel(
        input_size=input_size,
        hidden_size=params["units"],
        num_layers=params["layers"],
        dropout=params["dropout"]
    ).to(device)

    # Weighted BCE loss for class imbalance
    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for _ in range(max_epochs):
        model.train()

        # Shuffle training samples each epoch
        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            if len(idx) < 2:
                continue

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        # Validation loss for early stopping
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_count = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    # Load best validation-loss model
    if best_state is not None:
        model.load_state_dict(best_state)

    # Tune threshold on val
    model.eval()
    with torch.no_grad():
        y_val_prob = torch.sigmoid(model(X_va)).cpu().numpy()
    thr, _ = best_tss_threshold(y_val.astype(int), y_val_prob)

    # Prediction on the eval split
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= thr).astype(int)
    infer_time = time.time() - t0

    metrics = evaluate_model_metrics(
        y_actual=y_test.astype(int),
        y_pred=y_pred,
        y_prob=y_prob,
        model_name="LSTM"
    )
    metrics["threshold"] = thr
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

In [19]:
# ══════════════════════════════════════════════════════════════
# RUN LSTM EXPERIMENTS
# Tune on validation set, then evaluate best params on test set
# ══════════════════════════════════════════════════════════════
N_FINAL_RUNS = 3
N_VAL_SEEDS = 3

print(f"{'═' * 60}")
print("  Classifier : LSTM")
print(f"{'═' * 60}")

# Sanity check: LSTM expects (samples, 288, channels)
print("X_train:", X_train_norm.shape, " X_val:", X_val_norm.shape, " X_test:", X_test_norm.shape)

class_ratio = round((y_train == 0).sum() / (y_train == 1).sum(), 1)
print(f"Train class ratio 0:1 = {class_ratio}:1\n")

# Hyperparameter search on validation set only───────────────────
print(f"Searching {len(LSTM_GRID)} LSTM combinations on validation set...\n")

lstm_best_params = None
lstm_best_tss = -999
lstm_best_val_metrics = None

for i, params in enumerate(LSTM_GRID):
    print(f"[{i + 1}/{len(LSTM_GRID)}] {lstm_label(params)}", flush=True)

    # Run the same config over several seeds and average the results  
    val_runs = [] 
    for s in range(N_VAL_SEEDS):  
        # Validation set is used as the evaluation split during tuning
        metrics, _ = train_and_evaluate_lstm(
            params,
            X_train_norm, y_train,
            X_val_norm,   y_val,
            X_val_norm,   y_val,
            class_ratio=class_ratio,
            seed=RAND_STATE + s 
        )
        val_runs.append(metrics) 

        print( 
            f"    seed={RAND_STATE + s}  Val TSS={metrics['TSS']:.4f}  " 
            f"F1={metrics['F1-Score']:.4f}  " 
            f"Recall={metrics['Recall']:.4f}  "  
            f"Thr={metrics['threshold']:.4f}  "  
            f"Train={metrics['train_time']:.1f}s" 
        )  

    val_tss = [m["TSS"] for m in val_runs] 
    mean_val_tss = float(np.mean(val_tss))  
    std_val_tss = float(np.std(val_tss))  
    print(f"    Mean Val TSS={mean_val_tss:.4f}  std={std_val_tss:.4f}\n") 

    if mean_val_tss > lstm_best_tss: 
        lstm_best_tss = mean_val_tss 
        lstm_best_params = copy.deepcopy(params)
        lstm_best_val_metrics = val_runs   
        lstm_best_val_std = std_val_tss 

print("Best LSTM hyperparameters selected from validation set:")
print(f"  Params  : {lstm_label(lstm_best_params)}")
print(f"  Mean val TSS over {N_VAL_SEEDS} seeds: {lstm_best_tss:.4f}  std={lstm_best_val_std:.4f}")
for r, m in enumerate(lstm_best_val_metrics): 
    print(f"  Val seed {RAND_STATE + r}:") 
    print_metrics(m) 

# Final runs on the test set with the best hyperparameters───────
print(f"\nRunning {N_FINAL_RUNS} final runs with the best hyperparameters...\n")
print("─" * 70)
print(f"Params  : {lstm_label(lstm_best_params)}")
print(f"Ratio   : {class_ratio}:1")
print("─" * 70)

lstm_metrics_list = []

for run in range(N_FINAL_RUNS):
    print(f"Run {run + 1}...", flush=True)

    metrics, _ = train_and_evaluate_lstm(
        lstm_best_params,
        X_train_norm, y_train,
        X_val_norm,   y_val,
        X_test_norm,  y_test,
        class_ratio=class_ratio,
        seed=RAND_STATE + 100 + run 
    )

    lstm_metrics_list.append(metrics)

    print_metrics(metrics)

lstm_tss_runs = [m["TSS"] for m in lstm_metrics_list]
print(f"Test TSS over {N_FINAL_RUNS} runs: mean={np.mean(lstm_tss_runs):.4f}  std={np.std(lstm_tss_runs):.4f}")

print("\nLSTM experiment complete.")

════════════════════════════════════════════════════════════
  Classifier : LSTM
════════════════════════════════════════════════════════════
X_train: (12455, 288, 10)  X_val: (1780, 288, 10)  X_test: (3559, 288, 10)
Train class ratio 0:1 = 104.6:1

Searching 10 LSTM combinations on validation set...

[1/10] units=32  layers=1  drop=0.1  lr=0.001  bs=64
    seed=42  Val TSS=0.6308  F1=0.0977  Recall=0.7647  Thr=0.5990  Train=3.2s
    seed=43  Val TSS=0.6976  F1=0.1107  Recall=0.8235  Thr=0.6441  Train=3.8s
    seed=44  Val TSS=0.6046  F1=0.0671  Recall=0.8235  Thr=0.4785  Train=4.4s
    Mean Val TSS=0.6443  std=0.0392

[2/10] units=32  layers=1  drop=0.3  lr=0.001  bs=64
    seed=42  Val TSS=0.6286  F1=0.0963  Recall=0.7647  Thr=0.5935  Train=3.2s
    seed=43  Val TSS=0.6789  F1=0.0979  Recall=0.8235  Thr=0.5826  Train=3.8s
    seed=44  Val TSS=0.6112  F1=0.1224  Recall=0.7059  Thr=0.6889  Train=4.4s
    Mean Val TSS=0.6395  std=0.0287

[3/10] units=64  layers=1  drop=0.1  lr=0.001  bs

### RNN

In [20]:
# RNN model───────────────────────────────────────────────
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(RNNModel, self).__init__()

        self.rnn = nn.RNN(
            input_size,
            hidden_size,
            num_layers=num_layers,
            nonlinearity="tanh",
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.bn = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.bn(out)
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.dropout2(out)
        return self.fc2(out).squeeze(-1)


# RNN hyperparameter grid─────────────────────────────────
RNN_GRID = [
    # Small models
    {"units": 32,  "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 64},
    {"units": 32,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},

    # Medium models
    {"units": 64,  "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 64},
    {"units": 64,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},

    # Larger single-layer models
    {"units": 128, "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 32},
    {"units": 128, "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 32},

    # Two-layer models
    {"units": 64,  "dropout": 0.1, "layers": 2, "lr": 0.001,  "batch_size": 64},
    {"units": 128, "dropout": 0.3, "layers": 2, "lr": 0.001,  "batch_size": 32},

    # Lower learning rate variants
    {"units": 64,  "dropout": 0.3, "layers": 2, "lr": 0.0005, "batch_size": 64},
    {"units": 128, "dropout": 0.3, "layers": 2, "lr": 0.0005, "batch_size": 32},
]

def rnn_label(p):
    return (
        f"units={p['units']}  "
        f"layers={p['layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )

# Train RNN and evaluate on a provided test split──────────
def train_and_evaluate_rnn(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=50,
    patience=5,
    seed=RAND_STATE
):
    torch.manual_seed(seed)

    # Convert numpy arrays to tensors
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    # Build model
    input_size = X_train.shape[2]

    model = RNNModel(
        input_size=input_size,
        hidden_size=params["units"],
        num_layers=params["layers"],
        dropout=params["dropout"]
    ).to(device)

    # Weighted BCE loss for class imbalance
    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for _ in range(max_epochs):
        model.train()

        # Shuffle training samples each epoch
        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            if len(idx) < 2:
                continue

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            # Vanilla RNNs can have exploding gradients over 288 steps
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # Validation loss for early stopping
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_count = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    # Load best validation-loss model
    if best_state is not None:
        model.load_state_dict(best_state)

    # Tune threshold on val
    model.eval()
    with torch.no_grad():
        y_val_prob = torch.sigmoid(model(X_va)).cpu().numpy()
    thr, _ = best_tss_threshold(y_val.astype(int), y_val_prob)

    # Prediction on the eval split
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= thr).astype(int)
    infer_time = time.time() - t0

    metrics = evaluate_model_metrics(
        y_actual=y_test.astype(int),
        y_pred=y_pred,
        y_prob=y_prob,
        model_name="RNN"
    )
    metrics["threshold"] = thr
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

In [21]:
# ══════════════════════════════════════════════════════════════
# RUN RNN EXPERIMENTS
# Tune on validation set, then evaluate best params on test set
# ══════════════════════════════════════════════════════════════
N_FINAL_RUNS = 3
N_VAL_SEEDS = 3

print(f"{'═' * 60}")
print("  Classifier : RNN")
print(f"{'═' * 60}")

# Sanity check: RNN expects (samples, 288, channels)
print("X_train:", X_train_norm.shape, " X_val:", X_val_norm.shape, " X_test:", X_test_norm.shape)

class_ratio = round((y_train == 0).sum() / (y_train == 1).sum(), 1)
print(f"Train class ratio 0:1 = {class_ratio}:1\n")

# Hyperparameter search on validation set only───────────────────
print(f"Searching {len(RNN_GRID)} RNN combinations on validation set...\n")

rnn_best_params = None
rnn_best_tss = -999
rnn_best_val_metrics = None

for i, params in enumerate(RNN_GRID):
    print(f"[{i + 1}/{len(RNN_GRID)}] {rnn_label(params)}", flush=True)

    # Run the same config over several seeds and average the results
    val_runs = [] 
    for s in range(N_VAL_SEEDS):
        # Validation set is used as the evaluation split during tuning
        metrics, _ = train_and_evaluate_rnn(
            params,
            X_train_norm, y_train,
            X_val_norm,   y_val,
            X_val_norm,   y_val,
            class_ratio=class_ratio,
            seed=RAND_STATE + s 
        )
        val_runs.append(metrics)

        print(
            f"    seed={RAND_STATE + s}  Val TSS={metrics['TSS']:.4f}  "
            f"F1={metrics['F1-Score']:.4f}  " 
            f"Recall={metrics['Recall']:.4f}  " 
            f"Thr={metrics['threshold']:.4f}  "
            f"Train={metrics['train_time']:.1f}s"
        )

    val_tss = [m["TSS"] for m in val_runs] 
    mean_val_tss = float(np.mean(val_tss)) 
    std_val_tss = float(np.std(val_tss))
    print(f"    Mean Val TSS={mean_val_tss:.4f}  std={std_val_tss:.4f}\n")

    if mean_val_tss > rnn_best_tss: 
        rnn_best_tss = mean_val_tss  
        rnn_best_params = copy.deepcopy(params)
        rnn_best_val_metrics = val_runs 
        rnn_best_val_std = std_val_tss

print("Best RNN hyperparameters selected from validation set:")
print(f"  Params  : {rnn_label(rnn_best_params)}")
print(f"  Mean val TSS over {N_VAL_SEEDS} seeds: {rnn_best_tss:.4f}  std={rnn_best_val_std:.4f}")
for r, m in enumerate(rnn_best_val_metrics):
    print(f"  Val seed {RAND_STATE + r}:")
    print_metrics(m)

# Final runs on the test set with the best hyperparameters───────
print(f"\nRunning {N_FINAL_RUNS} final runs with the best hyperparameters...\n")
print("─" * 70)
print(f"Params  : {rnn_label(rnn_best_params)}")
print(f"Ratio   : {class_ratio}:1")
print("─" * 70)

rnn_metrics_list = []

for run in range(N_FINAL_RUNS):
    print(f"Run {run + 1}...", flush=True)

    metrics, _ = train_and_evaluate_rnn(
        rnn_best_params,
        X_train_norm, y_train,
        X_val_norm,   y_val,
        X_test_norm,  y_test,
        class_ratio=class_ratio,
        seed=RAND_STATE + 100 + run 
    )

    rnn_metrics_list.append(metrics)

    print_metrics(metrics)

rnn_tss_runs = [m["TSS"] for m in rnn_metrics_list]
print(f"Test TSS over {N_FINAL_RUNS} runs: mean={np.mean(rnn_tss_runs):.4f}  std={np.std(rnn_tss_runs):.4f}")

print("\nRNN experiment complete.")

════════════════════════════════════════════════════════════
  Classifier : RNN
════════════════════════════════════════════════════════════
X_train: (12455, 288, 10)  X_val: (1780, 288, 10)  X_test: (3559, 288, 10)
Train class ratio 0:1 = 104.6:1

Searching 10 RNN combinations on validation set...

[1/10] units=32  layers=1  drop=0.1  lr=0.001  bs=64
    seed=42  Val TSS=0.6834  F1=0.1007  Recall=0.8235  Thr=0.1849  Train=5.4s
    seed=43  Val TSS=0.4974  F1=0.0462  Recall=0.8235  Thr=0.1211  Train=3.5s
    seed=44  Val TSS=0.5123  F1=0.0547  Recall=0.7647  Thr=0.1717  Train=3.5s
    Mean Val TSS=0.5644  std=0.0844

[2/10] units=32  layers=1  drop=0.3  lr=0.001  bs=64
    seed=42  Val TSS=0.6254  F1=0.0619  Recall=0.8824  Thr=0.0702  Train=6.6s
    seed=43  Val TSS=0.5462  F1=0.0538  Recall=0.8235  Thr=0.1052  Train=3.6s
    seed=44  Val TSS=0.4544  F1=0.0760  Recall=0.5882  Thr=0.2565  Train=3.5s
    Mean Val TSS=0.5420  std=0.0699

[3/10] units=64  layers=1  drop=0.1  lr=0.001  bs=6

### CNN

In [22]:
# CNN model───────────────────────────────────────────────
class CNNModel(nn.Module):
    def __init__(self, input_size, filters=32, kernel_size=5, num_layers=2, dropout=0.3):
        super(CNNModel, self).__init__()

        blocks = []
        in_ch = input_size
        for _ in range(num_layers):
            blocks += [
                nn.Conv1d(in_ch, filters, kernel_size, padding=kernel_size // 2),
                nn.BatchNorm1d(filters),
                nn.ReLU(),
                nn.MaxPool1d(2)
            ]
            in_ch = filters
        self.conv = nn.Sequential(*blocks)

        self.pool = nn.AdaptiveAvgPool1d(1)     # Global average pool over time
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(filters, 32)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        out = x.transpose(1, 2)         # (batch, 288, C) -> (batch, C, 288) for Conv1d
        out = self.conv(out)
        out = self.pool(out).squeeze(-1)
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.dropout2(out)
        return self.fc2(out).squeeze(-1)


# CNN hyperparameter grid─────────────────────────────────
CNN_GRID = [
    # Small models
    {"filters": 16, "kernel": 5, "layers": 2, "dropout": 0.1, "lr": 0.001,  "batch_size": 64},
    {"filters": 16, "kernel": 5, "layers": 2, "dropout": 0.3, "lr": 0.001,  "batch_size": 64},

    # Medium models
    {"filters": 32, "kernel": 5, "layers": 2, "dropout": 0.1, "lr": 0.001,  "batch_size": 64},
    {"filters": 32, "kernel": 5, "layers": 2, "dropout": 0.3, "lr": 0.001,  "batch_size": 64},

    # Wider kernels (longer temporal patterns)
    {"filters": 32, "kernel": 9, "layers": 2, "dropout": 0.3, "lr": 0.001,  "batch_size": 64},
    {"filters": 32, "kernel": 3, "layers": 2, "dropout": 0.3, "lr": 0.001,  "batch_size": 64},

    # Larger models
    {"filters": 64, "kernel": 5, "layers": 2, "dropout": 0.3, "lr": 0.001,  "batch_size": 32},

    # Deeper models
    {"filters": 32, "kernel": 5, "layers": 3, "dropout": 0.3, "lr": 0.001,  "batch_size": 64},
    {"filters": 64, "kernel": 5, "layers": 3, "dropout": 0.3, "lr": 0.001,  "batch_size": 32},

    # Lower learning rate variant
    {"filters": 32, "kernel": 5, "layers": 3, "dropout": 0.3, "lr": 0.0005, "batch_size": 64},
]

def cnn_label(p):
    return (
        f"filters={p['filters']}  "
        f"kernel={p['kernel']}  "
        f"layers={p['layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )

# Train CNN and evaluate on a provided test split─────────
def train_and_evaluate_cnn(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=50,
    patience=5,
    seed=RAND_STATE
):
    torch.manual_seed(seed)

    # Convert numpy arrays to tensors
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    # Build model
    input_size = X_train.shape[2]

    model = CNNModel(
        input_size=input_size,
        filters=params["filters"],
        kernel_size=params["kernel"],
        num_layers=params["layers"],
        dropout=params["dropout"]
    ).to(device)

    # Weighted BCE loss for class imbalance
    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for _ in range(max_epochs):
        model.train()

        # Shuffle training samples each epoch
        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            if len(idx) < 2:
                continue

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        # Validation loss for early stopping
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_count = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    # Load best validation-loss model
    if best_state is not None:
        model.load_state_dict(best_state)

    # Tune threshold on val
    model.eval()
    with torch.no_grad():
        y_val_prob = torch.sigmoid(model(X_va)).cpu().numpy()
    thr, _ = best_tss_threshold(y_val.astype(int), y_val_prob)

    # Prediction on the eval split
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= thr).astype(int)
    infer_time = time.time() - t0

    metrics = evaluate_model_metrics(
        y_actual=y_test.astype(int),
        y_pred=y_pred,
        y_prob=y_prob,
        model_name="CNN"
    )
    metrics["threshold"] = thr
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

In [ ]:
# ══════════════════════════════════════════════════════════════
# RUN CNN EXPERIMENTS
# Tune on validation set, then evaluate best params on test set
# ══════════════════════════════════════════════════════════════
N_FINAL_RUNS = 3 
N_VAL_SEEDS = 3 

print(f"{'═' * 60}")
print("  Classifier : CNN")
print(f"{'═' * 60}")

# Sanity check: CNN expects (samples, 288, channels)
print("X_train:", X_train_norm.shape, " X_val:", X_val_norm.shape, " X_test:", X_test_norm.shape)

class_ratio = round((y_train == 0).sum() / (y_train == 1).sum(), 1)
print(f"Train class ratio 0:1 = {class_ratio}:1\n")

# Hyperparameter search on validation set only───────────────────
print(f"Searching {len(CNN_GRID)} CNN combinations on validation set...\n")

cnn_best_params = None
cnn_best_tss = -999
cnn_best_val_metrics = None

for i, params in enumerate(CNN_GRID):
    print(f"[{i + 1}/{len(CNN_GRID)}] {cnn_label(params)}", flush=True)

    # Run the same config over several seeds and average the results
    val_runs = [] 
    for s in range(N_VAL_SEEDS):
        # Validation set is used as the evaluation split during tuning
        metrics, _ = train_and_evaluate_cnn(
            params,
            X_train_norm, y_train,
            X_val_norm,   y_val,
            X_val_norm,   y_val,
            class_ratio=class_ratio,
            seed=RAND_STATE + s 
        )
        val_runs.append(metrics) 

        print( 
            f"    seed={RAND_STATE + s}  Val TSS={metrics['TSS']:.4f}  "
            f"F1={metrics['F1-Score']:.4f}  " 
            f"Recall={metrics['Recall']:.4f}  "  
            f"Thr={metrics['threshold']:.4f}  " 
            f"Train={metrics['train_time']:.1f}s" 
        ) 

    val_tss = [m["TSS"] for m in val_runs] 
    mean_val_tss = float(np.mean(val_tss)) 
    std_val_tss = float(np.std(val_tss)) 
    print(f"    Mean Val TSS={mean_val_tss:.4f}  std={std_val_tss:.4f}\n")

    if mean_val_tss > cnn_best_tss:
        cnn_best_tss = mean_val_tss 
        cnn_best_params = copy.deepcopy(params)
        cnn_best_val_metrics = val_runs
        cnn_best_val_std = std_val_tss

print("Best CNN hyperparameters selected from validation set:")
print(f"  Params  : {cnn_label(cnn_best_params)}")
print(f"  Mean val TSS over {N_VAL_SEEDS} seeds: {cnn_best_tss:.4f}  std={cnn_best_val_std:.4f}")
for r, m in enumerate(cnn_best_val_metrics):
    print(f"  Val seed {RAND_STATE + r}:")
    print_metrics(m)

# Final runs on the test set with the best hyperparameters───────
print(f"\nRunning {N_FINAL_RUNS} final runs with the best hyperparameters\n")
print("─" * 70)
print(f"Params  : {cnn_label(cnn_best_params)}")
print(f"Ratio   : {class_ratio}:1")
print("─" * 70)

cnn_metrics_list = []

for run in range(N_FINAL_RUNS):
    print(f"Run {run + 1}...", flush=True)

    metrics, _ = train_and_evaluate_cnn(
        cnn_best_params,
        X_train_norm, y_train,
        X_val_norm,   y_val,
        X_test_norm,  y_test,
        class_ratio=class_ratio,
        seed=RAND_STATE + 100 + run
    )

    cnn_metrics_list.append(metrics)

    print_metrics(metrics)

cnn_tss_runs = [m["TSS"] for m in cnn_metrics_list]
print(f"Test TSS over {N_FINAL_RUNS} runs: mean={np.mean(cnn_tss_runs):.4f}  std={np.std(cnn_tss_runs):.4f}")

print("\nCNN experiment complete.")

════════════════════════════════════════════════════════════
  Classifier : CNN
════════════════════════════════════════════════════════════
X_train: (12455, 288, 10)  X_val: (1780, 288, 10)  X_test: (3559, 288, 10)
Train class ratio 0:1 = 104.6:1

Searching 10 CNN combinations on validation set...

[1/10] filters=16  kernel=5  layers=2  drop=0.1  lr=0.001  bs=64
    seed=42  Val TSS=0.6294  F1=0.0628  Recall=0.8824  Thr=0.4469  Train=8.6s
    seed=43  Val TSS=0.6719  F1=0.0744  Recall=0.8824  Thr=0.3938  Train=6.2s
    seed=44  Val TSS=0.7116  F1=0.0901  Recall=0.8824  Thr=0.5299  Train=5.8s
    Mean Val TSS=0.6710  std=0.0336

[2/10] filters=16  kernel=5  layers=2  drop=0.3  lr=0.001  bs=64
    seed=42  Val TSS=0.6640  F1=0.0719  Recall=0.8824  Thr=0.4573  Train=6.9s
    seed=43  Val TSS=0.6645  F1=0.0721  Recall=0.8824  Thr=0.3958  Train=6.4s
    seed=44  Val TSS=0.7139  F1=0.0912  Recall=0.8824  Thr=0.5321  Train=5.9s
    Mean Val TSS=0.6808  std=0.0234

[3/10] filters=32  kernel=5